In [1]:
# Manipulation des données et chemins
from pathlib import Path
import numpy as np
import pandas as pd

# Modèle
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.base import clone

# Validation croisée et recherche d'hyperparamètres
from sklearn.model_selection import KFold, RandomizedSearchCV

# Métriques
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Visualisation et affichage
import matplotlib.pyplot as plt
from IPython.display import display

# Sauvegarde du modèle
import joblib

RANDOM_STATE = 42

In [6]:
# Charger les données préparées avec KNN
X_train = pd.read_csv("X_train_log_knn.csv", index_col="id")
X_test = pd.read_csv("X_test_log_knn.csv", index_col="id")

# Aligner les cibles sur les observations conservées, dans le même ordre
y = pd.read_csv("y_train.csv", index_col="id")["y"]
y_train = y.loc[X_train.index]

print("X_train :", X_train.shape)
print("y_train :", y_train.shape)
print("X_test  :", X_test.shape)
print("Indices alignés :", X_train.index.equals(y_train.index))

X_train : (1152, 216)
y_train : (1152,)
X_test  : (776, 216)
Indices alignés : True


In [7]:
# Validation croisée reproductible
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Modèle de départ
extra_trees = ExtraTreesRegressor(
    n_estimators=500,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Entraînement et évaluation sur chaque fold
train_scores = []
val_scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train), start=1):
    X_tr = X_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    model = clone(extra_trees)
    model.fit(X_tr, y_tr)

    train_r2 = r2_score(y_tr, model.predict(X_tr))
    val_r2 = r2_score(y_val, model.predict(X_val))

    train_scores.append(train_r2)
    val_scores.append(val_r2)

    print(f"Fold {fold} | R² train : {train_r2:.4f} | R² validation : {val_r2:.4f}")

print(f"\nR² train moyen      : {np.mean(train_scores):.4f}")
print(f"R² validation moyen : {np.mean(val_scores):.4f}")
print(f"Écart-type validation : {np.std(val_scores):.4f}")

Fold 1 | R² train : 1.0000 | R² validation : 0.6908
Fold 2 | R² train : 1.0000 | R² validation : 0.6456
Fold 3 | R² train : 1.0000 | R² validation : 0.6432
Fold 4 | R² train : 1.0000 | R² validation : 0.6265
Fold 5 | R² train : 1.0000 | R² validation : 0.6648

R² train moyen      : 1.0000
R² validation moyen : 0.6542
Écart-type validation : 0.0220


In [8]:
# Modèle : la parallélisation sera gérée par RandomizedSearchCV
extra_trees_tuning = ExtraTreesRegressor(
    n_estimators=500,
    random_state=RANDOM_STATE,
    n_jobs=1
)

# Hyperparamètres à explorer
param_distributions = {
    "max_depth": [None, 5, 10, 15, 20, 30],
    "min_samples_leaf": [1, 2, 4, 8, 16],
    "max_features": [0.3, 0.5, 0.7, 1.0]
}

search = RandomizedSearchCV(
    estimator=extra_trees_tuning,
    param_distributions=param_distributions,
    n_iter=40,
    scoring="r2",
    cv=cv,
    refit=True,
    return_train_score=True,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)

search.fit(X_train, y_train)

# Résultats des configurations testées
results = pd.DataFrame(search.cv_results_).sort_values(
    "mean_test_score", ascending=False
)

print("Meilleurs hyperparamètres :", search.best_params_)
print(f"R² CV baseline : {np.mean(val_scores):.4f}")
print(f"R² CV tuning   : {search.best_score_:.4f}")

display(
    results[
        [
            "param_max_depth",
            "param_min_samples_leaf",
            "param_max_features",
            "mean_train_score",
            "mean_test_score",
            "std_test_score"
        ]
    ].head(10).round(4)
)

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Meilleurs hyperparamètres : {'min_samples_leaf': 1, 'max_features': 1.0, 'max_depth': None}
R² CV baseline : 0.6542
R² CV tuning   : 0.6542


,param_max_depth,param_min_samples_leaf,param_max_features,mean_train_score,mean_test_score,std_test_score
26,None,1,1.0,1.0000,0.6542,0.0220
35,30,1,0.7,1.0000,0.6526,0.0215
7,None,1,0.7,1.0000,0.6526,0.0215
23,15,1,0.7,0.9998,0.6523,0.0217
28,15,2,1.0,0.9964,0.6490,0.0238
20,15,1,0.5,0.9997,0.6488,0.0222
33,20,2,1.0,0.9967,0.6486,0.0231
15,20,2,0.7,0.9961,0.6458,0.0221
12,None,2,0.7,0.9961,0.6456,0.0217
3,10,1,1.0,0.9813,0.6412,0.0250


Further parameter tuning

In [9]:
# Deux espaces : sans bootstrap et avec bootstrap
param_distributions_refined = [
    {
        "bootstrap": [False],
        "max_features": [0.6, 0.8, 0.9, 1.0],
        "min_samples_leaf": [1, 2, 3, 4],
        "min_samples_split": [2, 5, 10, 20],
        "max_depth": [None, 15, 25]
    },
    {
        "bootstrap": [True],
        "max_samples": [0.6, 0.8, 1.0],
        "max_features": [0.6, 0.8, 0.9, 1.0],
        "min_samples_leaf": [1, 2, 3, 4],
        "min_samples_split": [2, 5, 10, 20],
        "max_depth": [None, 15, 25]
    }
]

search_refined = RandomizedSearchCV(
    estimator=ExtraTreesRegressor(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=1
    ),
    param_distributions=param_distributions_refined,
    n_iter=60,
    scoring="r2",
    cv=cv,
    refit=True,
    return_train_score=True,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)

search_refined.fit(X_train, y_train)

# Tableau des meilleures configurations
results_refined = pd.DataFrame(
    search_refined.cv_results_
).sort_values("mean_test_score", ascending=False)

print(f"Première recherche : {search.best_score_:.5f}")
print(f"Nouvelle recherche : {search_refined.best_score_:.5f}")
print("Meilleurs paramètres :", search_refined.best_params_)

display(
    results_refined[
        [
            "param_bootstrap",
            "param_max_samples",
            "param_max_features",
            "param_min_samples_leaf",
            "param_min_samples_split",
            "param_max_depth",
            "mean_train_score",
            "mean_test_score",
            "std_test_score"
        ]
    ].head(10).round(5)
)

# Conserver le meilleur des deux modèles pour la suite
best_search = max(
    [search, search_refined],
    key=lambda s: s.best_score_
)

best_extra_trees = clone(best_search.best_estimator_).set_params(
    n_jobs=-1
)

print(f"\nMeilleur R² CV conservé : {best_search.best_score_:.5f}")

Fitting 5 folds for each of 60 candidates, totalling 300 fits
Première recherche : 0.65418
Nouvelle recherche : 0.65117
Meilleurs paramètres : {'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.9, 'max_depth': None, 'bootstrap': False}


,param_bootstrap,param_max_samples,param_max_features,param_min_samples_leaf,param_min_samples_split,param_max_depth,mean_train_score,mean_test_score,std_test_score
44,False,NaN,0.9,1,5,None,0.99813,0.65117,0.02371
6,False,NaN,0.9,1,5,15,0.99758,0.64998,0.02430
9,False,NaN,0.8,2,2,25,0.99630,0.64730,0.02198
30,False,NaN,1.0,3,2,15,0.98641,0.64481,0.02165
22,False,NaN,0.8,2,10,15,0.97382,0.63975,0.02259
37,False,NaN,0.6,1,10,15,0.97866,0.63933,0.02223
47,False,NaN,0.6,4,5,15,0.96041,0.63511,0.01895
42,False,NaN,0.8,4,10,None,0.95271,0.63302,0.02261
39,False,NaN,0.6,4,10,15,0.94652,0.63106,0.01994
46,False,NaN,0.9,2,20,None,0.91780,0.62918,0.02322



Meilleur R² CV conservé : 0.65418


Création du fichier oof_extra_trees pour comparer les erreurs de svr xgboost et extra trees

In [5]:
# Charger les folds partagés et les aligner sur X_train
stacking_dir = Path("Stacking")

folds = pd.read_csv(
    stacking_dir / "folds.csv", index_col="id"
)["fold"].loc[X_train.index]

# Reprendre les meilleurs hyperparamètres du tuning
estimator = clone(search.best_estimator_).set_params(n_jobs=-1)

# Initialiser les prédictions out-of-fold
oof = pd.Series(
    np.nan, index=X_train.index, name="extra_trees"
)

# Chaque observation est prédite par un modèle entraîné sur les autres folds
for f in sorted(folds.unique()):
    train = folds != f
    val = folds == f

    model = clone(estimator)
    model.fit(X_train.loc[train], y_train.loc[train])

    oof.loc[val] = model.predict(X_train.loc[val])

assert oof.notna().all(), "Certaines prédictions OOF sont manquantes"

print(f"Extra Trees OOF R² : {r2_score(y_train, oof):.4f}")

# Sauvegarder les OOF pour comparer les erreurs et faire le stacking
oof.to_csv(stacking_dir / "oof_extra_trees.csv")

# Entraîner le modèle final sur toutes les observations
model = clone(estimator)
model.fit(X_train, y_train)

# Prédire le test et sauvegarder
test_pred = pd.Series(
    model.predict(X_test),
    index=X_test.index,
    name="extra_trees"
)

test_pred.to_csv(stacking_dir / "test_extra_trees.csv")

Extra Trees OOF R² : 0.6450
